# Trabajando con Fechas y Horas
- Los datos de fecha/hora también se llaman datos **temporales**. Temporal significa "relativo al tiempo".

In [ ]:
import polars as pl

## Importar un Dataset con Fechas y Horas
- ISO8601 es un estándar/convención para representar una fecha/hora como cadena de texto. Polars soporta la mayoría de los formatos ISO8601.
- Polars importará las columnas de fecha/hora como cadenas de texto por defecto.
- La columna `clock_in` usa el formato `AAAA-MM-DD HH:MM:SS`.
- La columna `clock_out` usa el formato `AAAA-MM-DDTHH:MM:SS` (`T` es un separador entre la fecha y la hora).

In [ ]:
pl.read_csv("clock_in_times.csv")

employee_id,clock_in,clock_out
str,str,str
"""E001""","""2025-07-01 08:55:00""","""2025-07-01T17:05:00"""
"""E002""","""2025-07-01 09:10:00""","""2025-07-01T17:45:00"""
"""E003""","""2025-07-01 08:50:00""","""2025-07-01T16:30:00"""
"""E004""","""2025-07-01 10:00:00""","""2025-07-01T18:00:00"""
"""E005""","""2025-07-01 07:45:00""","""2025-07-01T15:15:00"""


- Pasa `True` al parámetro `try_parse_dates` de la función `read_csv` para intentar parsear los valores de fecha/hora.
- Polars recurrirá a cadenas de texto si no puede convertir los valores de una columna a fechas/horas.
- El símbolo `[μs]` (mu) significa "precisión de microsegundos". Hay 1,000,000 de microsegundos en un segundo.

In [ ]:
pl.read_csv("clock_in_times.csv", try_parse_dates=True)

employee_id,clock_in,clock_out
str,datetime[μs],datetime[μs]
"""E001""",2025-07-01 08:55:00,2025-07-01 17:05:00
"""E002""",2025-07-01 09:10:00,2025-07-01 17:45:00
"""E003""",2025-07-01 08:50:00,2025-07-01 16:30:00
"""E004""",2025-07-01 10:00:00,2025-07-01 18:00:00
"""E005""",2025-07-01 07:45:00,2025-07-01 15:15:00


- Como alternativa, usa el parámetro `schema_overrides` para convertir columnas específicas a diferentes tipos de datos.
- El tipo `pl.Datetime` representa una fecha/hora.

In [ ]:
pl.read_csv(
    "clock_in_times.csv",
    schema_overrides={"clock_in": pl.Datetime, "clock_out": pl.Datetime},
)

employee_id,clock_in,clock_out
str,datetime[μs],datetime[μs]
"""E001""",2025-07-01 08:55:00,2025-07-01 17:05:00
"""E002""",2025-07-01 09:10:00,2025-07-01 17:45:00
"""E003""",2025-07-01 08:50:00,2025-07-01 16:30:00
"""E004""",2025-07-01 10:00:00,2025-07-01 18:00:00
"""E005""",2025-07-01 07:45:00,2025-07-01 15:15:00


- Si el `DataFrame` ya está creado, usa el método `str.to_datetime` para convertirlo en una columna de fecha/hora.

In [ ]:
clock_in_times = pl.read_csv("clock_in_times.csv")
clock_in_times

employee_id,clock_in,clock_out
str,str,str
"""E001""","""2025-07-01 08:55:00""","""2025-07-01T17:05:00"""
"""E002""","""2025-07-01 09:10:00""","""2025-07-01T17:45:00"""
"""E003""","""2025-07-01 08:50:00""","""2025-07-01T16:30:00"""
"""E004""","""2025-07-01 10:00:00""","""2025-07-01T18:00:00"""
"""E005""","""2025-07-01 07:45:00""","""2025-07-01T15:15:00"""


In [ ]:
clock_in_times.with_columns(
    pl.col("clock_in").str.to_datetime(), pl.col("clock_out").str.to_datetime()
)

employee_id,clock_in,clock_out
str,datetime[μs],datetime[μs]
"""E001""",2025-07-01 08:55:00,2025-07-01 17:05:00
"""E002""",2025-07-01 09:10:00,2025-07-01 17:45:00
"""E003""",2025-07-01 08:50:00,2025-07-01 16:30:00
"""E004""",2025-07-01 10:00:00,2025-07-01 18:00:00
"""E005""",2025-07-01 07:45:00,2025-07-01 15:15:00


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/api/polars.read_csv.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.str.to_datetime.html

## Parsear Fechas/Horas con el Método strptime
- El parámetro `try_parse_dates` puede fallar al convertir una columna de fecha/hora de cadenas de texto.
- Polars no podrá convertir las columnas `us_format` y `custom_format` a continuación.
- Si la conversión falla, importa la columna como cadenas de texto, luego usa el método `str.strptime` para convertir.
- Pasa el tipo de dato Polars deseado al parámetro `dtype`.

In [ ]:
weird_datetimes = pl.read_csv("weird_datetimes.csv", try_parse_dates=True)
weird_datetimes.head(2)

iso_format,us_format,custom_format
datetime[μs],str,str
2025-07-06 14:30:00,"""07/06/2025 02:30 PM""","""06-Jul-2025 14:30"""
2025-07-07 09:15:00,"""07/07/2025 09:15 AM""","""07-Jul-2025 09:15"""


- `strptime` significa "string - parse time" (es decir, parsear/leer una fecha/hora desde una cadena de texto).
- El método acepta una cadena de formato que usa símbolos para designar los componentes del formato de fecha/hora.
- Por ejemplo, el símbolo `%m` designa un mes, el símbolo `%d` designa un día, y el símbolo `%Y` designa un año de 4 dígitos.
- Incluye espacios en la cadena de formato. Debe coincidir perfectamente con el formato de la cadena de fecha/hora.
- El método `strptime` se basa en el crate `chrono` de Rust internamente.
- La sintaxis de la cadena de formato debe seguir el estándar del crate `chrono`, que puede diferir del estándar de Python.

In [ ]:
weird_datetimes.with_columns(
    pl.col("us_format").str.strptime(dtype=pl.Datetime, format="%m/%d/%Y %I:%M %p"),
    pl.col("custom_format").str.strptime(dtype=pl.Datetime, format="%d-%b-%Y %H:%M"),
)

iso_format,us_format,custom_format
datetime[μs],datetime[μs],datetime[μs]
2025-07-06 14:30:00,2025-07-06 14:30:00,2025-07-06 14:30:00
2025-07-07 09:15:00,2025-07-07 09:15:00,2025-07-07 09:15:00
2025-07-08 17:45:00,2025-07-08 17:45:00,2025-07-08 17:45:00
2025-07-09 23:59:59,2025-07-09 23:59:00,2025-07-09 23:59:00
2025-07-10 00:00:00,2025-07-10 00:00:00,2025-07-10 00:00:00


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.str.strptime.html

## Parsear Fechas y Horas

In [ ]:
weird_datetimes = pl.read_csv("weird_datetimes.csv", try_parse_dates=True)
weird_datetimes.head(2)

iso_format,us_format,custom_format
datetime[μs],str,str
2025-07-06 14:30:00,"""07/06/2025 02:30 PM""","""06-Jul-2025 14:30"""
2025-07-07 09:15:00,"""07/07/2025 09:15 AM""","""07-Jul-2025 09:15"""


- Pasa un tipo `pl.Date` para extraer solo la información de fecha (sin hora asociada).
- El parámetro `format` aún debe recibir la cadena de formato completa para poder parsear la cadena.

In [ ]:
weird_datetimes.with_columns(
    pl.col("us_format").str.strptime(dtype=pl.Date, format="%m/%d/%Y %I:%M %p"),
    pl.col("custom_format").str.strptime(dtype=pl.Date, format="%d-%b-%Y %H:%M"),
)

iso_format,us_format,custom_format
datetime[μs],date,date
2025-07-06 14:30:00,2025-07-06,2025-07-06
2025-07-07 09:15:00,2025-07-07,2025-07-07
2025-07-08 17:45:00,2025-07-08,2025-07-08
2025-07-09 23:59:59,2025-07-09,2025-07-09
2025-07-10 00:00:00,2025-07-10,2025-07-10


- Usa un `dtype` de `pl.Time` para extraer una columna de `time` (sin información de fecha).

In [ ]:
weird_datetimes.with_columns(
    pl.col("us_format").str.strptime(dtype=pl.Time, format="%m/%d/%Y %I:%M %p"),
    pl.col("custom_format").str.strptime(dtype=pl.Time, format="%d-%b-%Y %H:%M"),
)

iso_format,us_format,custom_format
datetime[μs],time,time
2025-07-06 14:30:00,14:30:00,14:30:00
2025-07-07 09:15:00,09:15:00,09:15:00
2025-07-08 17:45:00,17:45:00,17:45:00
2025-07-09 23:59:59,23:59:00,23:59:00
2025-07-10 00:00:00,00:00:00,00:00:00


- Como alternativa a `strptime` y `dtype`, usa los métodos `str.to_datetime`, `str.to_date` y `str.to_time`.
- Los métodos aceptan el formato de la cadena de tiempo.

In [ ]:
weird_datetimes.select(
    pl.col("us_format"),
    pl.col("us_format").str.to_datetime("%m/%d/%Y %I:%M %p").alias("datetime"),
    pl.col("us_format").str.to_date("%m/%d/%Y %I:%M %p").alias("date"),
    pl.col("us_format").str.to_time("%m/%d/%Y %I:%M %p").alias("time"),
)

us_format,datetime,date,time
str,datetime[μs],date,time
"""07/06/2025 02:30 PM""",2025-07-06 14:30:00,2025-07-06,14:30:00
"""07/07/2025 09:15 AM""",2025-07-07 09:15:00,2025-07-07,09:15:00
"""07/08/2025 05:45 PM""",2025-07-08 17:45:00,2025-07-08,17:45:00
"""07/09/2025 11:59 PM""",2025-07-09 23:59:00,2025-07-09,23:59:00
"""07/10/2025 12:00 AM""",2025-07-10 00:00:00,2025-07-10,00:00:00


### Lectura Adicional
- https://docs.pola.rs/user-guide/transformations/time-series/parsing/#casting-strings-to-dates
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.str.strptime.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.str.to_datetime.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.str.to_date.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.str.to_time.html

## Convertir de Fecha a Cadena de Texto
- El tipo `Datetime` maneja valores de fecha/hora (tanto un día calendario como una hora específica).
- El tipo `Date` representa una fecha calendario sin una hora asociada.
- El tipo `Time` representa una hora sin una fecha calendario asociada.
- El atributo/namespace `dt` contiene todos los métodos para construir expresiones temporales.

In [ ]:
clock_in_times = pl.read_csv("clock_in_times.csv", try_parse_dates=True)
clock_in_times

employee_id,clock_in,clock_out
str,datetime[μs],datetime[μs]
"""E001""",2025-07-01 08:55:00,2025-07-01 17:05:00
"""E002""",2025-07-01 09:10:00,2025-07-01 17:45:00
"""E003""",2025-07-01 08:50:00,2025-07-01 16:30:00
"""E004""",2025-07-01 10:00:00,2025-07-01 18:00:00
"""E005""",2025-07-01 07:45:00,2025-07-01 15:15:00


- El método `dt.date` extrae la fecha de una columna de fecha/hora.
- El método `dt.time` extrae la hora de una columna de fecha/hora.

In [ ]:
clock_in_times.with_columns(
    pl.col("clock_in").dt.date().alias("clock_in_date"),
    pl.col("clock_in").dt.time().alias("clock_in_time"),
)

employee_id,clock_in,clock_out,clock_in_date,clock_in_time
str,datetime[μs],datetime[μs],date,time
"""E001""",2025-07-01 08:55:00,2025-07-01 17:05:00,2025-07-01,08:55:00
"""E002""",2025-07-01 09:10:00,2025-07-01 17:45:00,2025-07-01,09:10:00
"""E003""",2025-07-01 08:50:00,2025-07-01 16:30:00,2025-07-01,08:50:00
"""E004""",2025-07-01 10:00:00,2025-07-01 18:00:00,2025-07-01,10:00:00
"""E005""",2025-07-01 07:45:00,2025-07-01 15:15:00,2025-07-01,07:45:00


- Las columnas de fecha/hora permiten operaciones temporales que serían imposibles con cadenas de texto.
- Por ejemplo, las fechas/horas soportan sumar o restar duraciones como columnas.
- Usa el método `dt.to_string` para convertir columnas de fecha/hora a cadenas de texto.
- Polars convertirá la fecha/hora a una cadena de texto con el estándar ISO 8601.

In [ ]:
clock_in_times.select(
    pl.col("clock_in"),
    pl.col("clock_in").dt.to_string().alias("default"),
    pl.col("clock_in").dt.to_string("%B %m, %Y").alias("formatted"),
    pl.col("clock_in").dt.date().dt.to_string().alias("date_default"),
    pl.col("clock_in").dt.date().dt.to_string("%B %m, %Y").alias("date_formatted"),
)

clock_in,default,formatted,date_default,date_formatted
datetime[μs],str,str,str,str
2025-07-01 08:55:00,"""2025-07-01 08:55:00.000000""","""July 07, 2025""","""2025-07-01""","""July 07, 2025"""
2025-07-01 09:10:00,"""2025-07-01 09:10:00.000000""","""July 07, 2025""","""2025-07-01""","""July 07, 2025"""
2025-07-01 08:50:00,"""2025-07-01 08:50:00.000000""","""July 07, 2025""","""2025-07-01""","""July 07, 2025"""
2025-07-01 10:00:00,"""2025-07-01 10:00:00.000000""","""July 07, 2025""","""2025-07-01""","""July 07, 2025"""
2025-07-01 07:45:00,"""2025-07-01 07:45:00.000000""","""July 07, 2025""","""2025-07-01""","""July 07, 2025"""


### Lectura Adicional
- https://docs.pola.rs/user-guide/expressions/casting/#parsing-formatting-temporal-data-types
- https://docs.pola.rs/user-guide/transformations/time-series/parsing/#extracting-date-features-from-a-date-column
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.date.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.time.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.to_string.html

## Extracción de Componentes de Fecha/Hora
- El atributo `dt` contiene varios métodos para extraer componentes de fecha (día, mes, año, etc.)

In [ ]:
history = pl.read_csv("history.csv", try_parse_dates=True)
history.head(3)

event,date
str,date
"""Declaration of Independence""",1776-07-04
"""Constitution Signed""",1787-09-17
"""Louisiana Purchase""",1803-04-30


- El método `dt.millennium` devuelve el milenio. Un milenio es un período de 1000 años.
- El método `dt.century` devuelve el siglo. Un siglo es un período de 100 años.
- El método `dt.year` devuelve el año.
- El método `dt.month` devuelve el mes.
- El método `dt.day` devuelve el día.
- El método `dt.quarter` devuelve el trimestre del año.

In [ ]:
history.with_columns(
    pl.col("date").dt.millennium().alias("millennium"),
    pl.col("date").dt.century().alias("century"),
    pl.col("date").dt.year().alias("year"),
    pl.col("date").dt.month().alias("month"),
    pl.col("date").dt.day().alias("day"),
    pl.col("date").dt.quarter().alias("quarter"),
)

event,date,millennium,century,year,month,day,quarter
str,date,i32,i32,i32,i8,i8,i8
"""Declaration of Independence""",1776-07-04,2,18,1776,7,4,3
"""Constitution Signed""",1787-09-17,2,18,1787,9,17,3
"""Louisiana Purchase""",1803-04-30,2,19,1803,4,30,2
"""Civil War Begins""",1861-04-12,2,19,1861,4,12,2
"""Emancipation Proclamation""",1863-01-01,2,19,1863,1,1,1
"""Pearl Harbor Attack""",1941-12-07,2,20,1941,12,7,4
"""Moon Landing""",1969-07-20,2,20,1969,7,20,3
"""Release of Polars Course""",2025-09-30,3,21,2025,9,30,3


- El método `dt.weekday` devuelve el día de la semana como un número. El lunes es 1 y el domingo es 7.
- El método `dt.days_in_month` devuelve el número de días en el mes de la fecha.
- El método `dt.ordinal_day` devuelve el día del año.

In [ ]:
history.with_columns(
    pl.col("date").dt.weekday().alias("weekday"),
    pl.col("date").dt.days_in_month().alias("days_in_month"),
    pl.col("date").dt.ordinal_day().alias("ordinal_day"),
)

event,date,weekday,days_in_month,ordinal_day
str,date,i8,i8,i16
"""Declaration of Independence""",1776-07-04,4,31,186
"""Constitution Signed""",1787-09-17,1,30,260
"""Louisiana Purchase""",1803-04-30,6,30,120
"""Civil War Begins""",1861-04-12,5,30,102
"""Emancipation Proclamation""",1863-01-01,4,31,1
"""Pearl Harbor Attack""",1941-12-07,7,31,341
"""Moon Landing""",1969-07-20,7,31,201
"""Release of Polars Course""",2025-09-30,2,30,273


- `dt.is_business_day` devuelve True si el día es un día laboral (de lunes a viernes).
- `dt.is_leap_year` devuelve True si el año es un año bisiesto.

In [ ]:
history.with_columns(
    pl.col("date").dt.is_business_day().alias("is_business_day"),
    pl.col("date").dt.is_leap_year().alias("is_leap_year"),
)

event,date,is_business_day,is_leap_year
str,date,bool,bool
"""Declaration of Independence""",1776-07-04,true,true
"""Constitution Signed""",1787-09-17,true,false
"""Louisiana Purchase""",1803-04-30,false,false
"""Civil War Begins""",1861-04-12,true,false
"""Emancipation Proclamation""",1863-01-01,true,false
"""Pearl Harbor Attack""",1941-12-07,false,false
"""Moon Landing""",1969-07-20,false,false
"""Release of Polars Course""",2025-09-30,true,false


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.millennium.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.century.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.year.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.month.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.day.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.quarter.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.weekday.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.days_in_month.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.ordinal_day.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.is_business_day.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.is_leap_year.html

## Filtrar por Fecha, Hora y Fecha/Hora
- Polars soporta fechas, horas y fechas/horas en expresiones.
- Usa las funciones `pl.date`, `pl.time` y `pl.datetime` para modelar el valor temporal con el que comparar cada fila.
- Estas son distintas de los _tipos_ `pl.Date`, `pl.Time` y `pl.Datetime`.
- Los objetos `date`, `datetime` y `time` del módulo `datetime` de Python también funcionan.

In [ ]:
history = pl.read_csv("history.csv", try_parse_dates=True)
history

event,date
str,date
"""Declaration of Independence""",1776-07-04
"""Constitution Signed""",1787-09-17
"""Louisiana Purchase""",1803-04-30
"""Civil War Begins""",1861-04-12
"""Emancipation Proclamation""",1863-01-01
"""Pearl Harbor Attack""",1941-12-07
"""Moon Landing""",1969-07-20
"""Release of Polars Course""",2025-09-30


In [ ]:
history.filter(pl.col("date").dt.is_business_day())

event,date
str,date
"""Declaration of Independence""",1776-07-04
"""Constitution Signed""",1787-09-17
"""Civil War Begins""",1861-04-12
"""Emancipation Proclamation""",1863-01-01
"""Release of Polars Course""",2025-09-30


In [ ]:
history.filter(pl.col("date") == pl.date(1803, 4, 30))
# history.filter(pl.col("date").eq(pl.date(1803, 4, 30)))

event,date
str,date
"""Louisiana Purchase""",1803-04-30


In [ ]:
import datetime as dt

In [ ]:
history.filter(pl.col("date") > dt.date(1800, 1, 1))
# history.filter(pl.col("date").gt(dt.date(1800, 1, 1)))

event,date
str,date
"""Louisiana Purchase""",1803-04-30
"""Civil War Begins""",1861-04-12
"""Emancipation Proclamation""",1863-01-01
"""Pearl Harbor Attack""",1941-12-07
"""Moon Landing""",1969-07-20
"""Release of Polars Course""",2025-09-30


- El método `is_between` es ideal para extraer fechas que caen dentro de un rango de tiempo.
- Ambos extremos son inclusivos.

In [ ]:
history.filter(pl.col("date").is_between(pl.date(1800, 1, 1), pl.date(1899, 12, 31)))
# history.filter(pl.col("date").dt.century().eq(19))

event,date
str,date
"""Louisiana Purchase""",1803-04-30
"""Civil War Begins""",1861-04-12
"""Emancipation Proclamation""",1863-01-01


- Los mismos métodos aplican para fechas/horas y horas.

In [ ]:
clock_in_times = (
    pl.read_csv("clock_in_times.csv", try_parse_dates=True)
    .select("clock_in")
    .sort("clock_in")
)
clock_in_times

clock_in
datetime[μs]
2025-07-01 07:45:00
2025-07-01 08:50:00
2025-07-01 08:55:00
2025-07-01 09:10:00
2025-07-01 10:00:00


In [ ]:
clock_in_times.filter(pl.col("clock_in") >= pl.datetime(2025, 7, 1, 8, 53, 0))
# clock_in_times.filter(pl.col("clock_in").ge(pl.datetime(2025, 7, 1, 8, 53, 0)))

clock_in
datetime[μs]
2025-07-01 08:55:00
2025-07-01 09:10:00
2025-07-01 10:00:00


In [ ]:
clock_in_times.select(pl.col("clock_in").cast(pl.Time)).filter(
    pl.col("clock_in") < (pl.time(9, 0, 0))
)

clock_in
time
07:45:00
08:50:00
08:55:00


### Lectura Adicional
- https://docs.pola.rs/user-guide/transformations/time-series/filter/#filtering-by-single-dates
- https://docs.pola.rs/user-guide/transformations/time-series/filter/#filtering-by-a-date-range
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.is_business_day.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.datetime
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.date

## Sumar y Restar Tiempo I
- Una duración/timedelta modela un intervalo de tiempo (por ejemplo, 5 horas, 40 minutos).

In [ ]:
history = pl.read_csv("history.csv", try_parse_dates=True)
history.head(2)

event,date
str,date
"""Declaration of Independence""",1776-07-04
"""Constitution Signed""",1787-09-17


- Usa el signo `+` para sumar tiempo y el signo `-` para restar tiempo.
- Los objetos `dt.timedelta` y `pl.duration` representan una duración.
- La función `pl.duration` acepta parámetros `days`, `weeks`, `hours`, y más.

### Diferencias entre `pl.duration` y `dt.timedelta`

| Característica | `pl.duration` (Polars) | `datetime.timedelta` (Python) |
| :--- | :--- | :--- |
| **Origen** | Nativo de Polars (basado en Rust) | Biblioteca estándar de Python |
| **Uso principal** | Expresiones y columnas de Polars | Lógica general de Python |
| **Unidades** | Semanas, días, horas, minutos, segundos, milisegundos, microsegundos, nanosegundos | Semanas, días, horas, minutos, segundos, milisegundos, microsegundos |
| **Rendimiento** | Optimizado para procesamiento vectorial | Requiere conversión al usarse en DataFrames |

In [ ]:
history.with_columns((pl.col("date") + pl.duration(days=3)).alias("new_date"))
# history.with_columns(pl.col("date").add(pl.duration(days=3)).alias("new_date"))
# history.with_columns(pl.col("date").add(pl.duration(weeks=8, days=3)).alias("new_date"))

event,date,new_date
str,date,date
"""Declaration of Independence""",1776-07-04,1776-07-07
"""Constitution Signed""",1787-09-17,1787-09-20
"""Louisiana Purchase""",1803-04-30,1803-05-03
"""Civil War Begins""",1861-04-12,1861-04-15
"""Emancipation Proclamation""",1863-01-01,1863-01-04
"""Pearl Harbor Attack""",1941-12-07,1941-12-10
"""Moon Landing""",1969-07-20,1969-07-23
"""Release of Polars Course""",2025-09-30,2025-10-03


In [ ]:
history.with_columns((pl.col("date") - pl.duration(days=3)).alias("new_date"))
# history.with_columns(pl.col("date").sub(pl.duration(days=3)).alias("new_date"))

event,date,new_date
str,date,date
"""Declaration of Independence""",1776-07-04,1776-07-01
"""Constitution Signed""",1787-09-17,1787-09-14
"""Louisiana Purchase""",1803-04-30,1803-04-27
"""Civil War Begins""",1861-04-12,1861-04-09
"""Emancipation Proclamation""",1863-01-01,1862-12-29
"""Pearl Harbor Attack""",1941-12-07,1941-12-04
"""Moon Landing""",1969-07-20,1969-07-17
"""Release of Polars Course""",2025-09-30,2025-09-27


- Polars mantendrá el tipo de dato de la columna original.
- Por lo tanto, ignorará horas, minutos y segundos para una columna de fecha.

In [ ]:
history.with_columns(
    pl.col("date").add(pl.duration(weeks=8, days=3, hours=5)).alias("new_date")
)

event,date,new_date
str,date,date
"""Declaration of Independence""",1776-07-04,1776-09-01
"""Constitution Signed""",1787-09-17,1787-11-15
"""Louisiana Purchase""",1803-04-30,1803-06-28
"""Civil War Begins""",1861-04-12,1861-06-10
"""Emancipation Proclamation""",1863-01-01,1863-03-01
"""Pearl Harbor Attack""",1941-12-07,1942-02-04
"""Moon Landing""",1969-07-20,1969-09-17
"""Release of Polars Course""",2025-09-30,2025-11-28


- Convierte la columna a fecha/hora _primero_, luego realiza la suma.

In [ ]:
history.with_columns(
    pl.col("date")
    .cast(pl.Datetime)
    .add(pl.duration(weeks=8, days=3, hours=5))
    .alias("new_date")
)

event,date,new_date
str,date,datetime[μs]
"""Declaration of Independence""",1776-07-04,1776-09-01 05:00:00
"""Constitution Signed""",1787-09-17,1787-11-15 05:00:00
"""Louisiana Purchase""",1803-04-30,1803-06-28 05:00:00
"""Civil War Begins""",1861-04-12,1861-06-10 05:00:00
"""Emancipation Proclamation""",1863-01-01,1863-03-01 05:00:00
"""Pearl Harbor Attack""",1941-12-07,1942-02-04 05:00:00
"""Moon Landing""",1969-07-20,1969-09-17 05:00:00
"""Release of Polars Course""",2025-09-30,2025-11-28 05:00:00


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.duration
- https://docs.pola.rs/api/python/stable/reference/api/polars.datatypes.Datetime.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.add.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.sub.html

## Sumar y Restar Tiempo II
- Se recomienda el método `dt.offset_by` para sumar/restar fechas/horas.
- A diferencia del tipo duración, soporta años/meses y tiene en cuenta matices como los años bisiestos.
- El especificador `%D` es un atajo para `%m/%d/%y` (mes/día/año).

In [ ]:
deliveries = pl.read_csv("deliveries.csv").select(
    pl.col("order_date").str.to_datetime(format="%D")
)
deliveries.head(2)

order_date
datetime[μs]
1998-05-24 00:00:00
1992-04-22 00:00:00


- El método `dt.offset_by` acepta su propia cadena de formato.
- `h` significa hora, `m` significa minuto, y así sucesivamente.

In [ ]:
deliveries.with_columns(
    pl.col("order_date").dt.offset_by("-12h30m").alias("time_in_12_and_a_half_hours")
)

order_date,time_in_12_and_a_half_hours
datetime[μs],datetime[μs]
1998-05-24 00:00:00,1998-05-23 11:30:00
1992-04-22 00:00:00,1992-04-21 11:30:00
1991-02-10 00:00:00,1991-02-09 11:30:00
1992-07-21 00:00:00,1992-07-20 11:30:00
1993-09-02 00:00:00,1993-09-01 11:30:00
…,…
1991-06-24 00:00:00,1991-06-23 11:30:00
1991-09-09 00:00:00,1991-09-08 11:30:00
1990-11-16 00:00:00,1990-11-15 11:30:00


- La ventaja de `dt.offset_by` es su mayor conocimiento del tiempo.
- Por ejemplo, supongamos que queremos encontrar el mismo día calendario del mes siguiente.
- No podemos sumar una duración constante porque los meses tienen diferente número de días.
- El símbolo `mo` significa "mes calendario".

In [ ]:
deliveries.with_columns(
    pl.col("order_date").dt.offset_by("1mo").alias("same_calendar_day_next_month")
)

order_date,same_calendar_day_next_month
datetime[μs],datetime[μs]
1998-05-24 00:00:00,1998-06-24 00:00:00
1992-04-22 00:00:00,1992-05-22 00:00:00
1991-02-10 00:00:00,1991-03-10 00:00:00
1992-07-21 00:00:00,1992-08-21 00:00:00
1993-09-02 00:00:00,1993-10-02 00:00:00
…,…
1991-06-24 00:00:00,1991-07-24 00:00:00
1991-09-09 00:00:00,1991-10-09 00:00:00
1990-11-16 00:00:00,1990-12-16 00:00:00


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.offset_by.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.str.to_datetime.html

## El Tipo duration
- El tipo duración de Polars representa un intervalo de tiempo.
- Ordenar duraciones funciona como se espera. Un orden ascendente significa de "duración más corta" a "duración más larga".

In [ ]:
deliveries = pl.read_csv("deliveries.csv").with_columns(
    pl.col("order_date", "delivery_date").str.to_date(format="%D")
)
deliveries.head(2)

ID,order_date,delivery_date
i64,date,date
1,1998-05-24,1999-02-05
2,1992-04-22,1998-03-06


In [ ]:
deliveries_with_durations = deliveries.with_columns(
    (pl.col("delivery_date") - pl.col("order_date")).alias("time_to_deliver")
).sort("time_to_deliver")

In [ ]:
deliveries_with_durations.schema

Schema([('ID', Int64),
        ('order_date', Date),
        ('delivery_date', Date),
        ('time_to_deliver', Duration(time_unit='us'))])

- Los métodos para manipular valores tipo `duration` también se encuentran dentro del atributo `dt`.
- La familia de métodos `dt.total_` representa la duración con una unidad de tiempo diferente.
- Por ejemplo, `dt.total_hours` representa la duración en horas (como un `i64`).

In [ ]:
deliveries_with_durations.with_columns(
    pl.col("time_to_deliver").dt.total_days().alias("total_days"),
    pl.col("time_to_deliver").dt.total_hours().alias("total_hours"),
    pl.col("time_to_deliver").dt.total_minutes().alias("total_minutes"),
)

ID,order_date,delivery_date,time_to_deliver,total_days,total_hours,total_minutes
i64,date,date,duration[μs],i64,i64,i64
898,1990-05-24,1990-06-01,8d,8,192,11520
19,1998-05-10,1998-05-19,9d,9,216,12960
612,1994-08-11,1994-08-20,9d,9,216,12960
994,1993-06-03,1993-06-13,10d,10,240,14400
310,1997-09-20,1997-10-06,16d,16,384,23040
…,…,…,…,…,…,…
331,1990-09-18,1999-12-19,3379d,3379,81096,4865760
130,1990-04-02,1999-08-16,3423d,3423,82152,4929120
904,1990-02-13,1999-11-15,3562d,3562,85488,5129280


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.total_days.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.total_hours.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.total_minutes.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.str.to_date.html
- https://docs.pola.rs/api/python/stable/reference/api/polars.datatypes.Duration.html

## Zonas Horarias I
- El Tiempo Universal Coordinado (UTC) establece un punto de referencia/estándar para la hora actual.
- Una fecha/hora "naive" (ingenua) es una que no tiene conocimiento de la zona horaria. Todas las fechas/horas que hemos visto hasta ahora han sido naive.
- El método `read_json` NO soporta el parámetro `try_parse_dates`.

In [ ]:
pl.read_json("flights.json")

flight_id,airline,aircraft,status,departure_city,departure_time,arrival_city,arrival_time
str,str,str,str,str,str,str,str
"""AA593""","""American Airlines""","""Boeing 787""","""Delayed""","""Paris""","""2025-06-22 21:00:00""","""Seattle""","""2025-06-23 01:00:00"""
"""AA885""","""Lufthansa""","""Airbus A320""","""On Time""","""Denver""","""2025-06-18 19:00:00""","""New York""","""2025-06-18 23:00:00"""
"""UA303""","""United""","""Boeing 737""","""Delayed""","""Miami""","""2025-06-16 23:00:00""","""Denver""","""2025-06-17 04:00:00"""
"""AA602""","""Delta""","""Boeing 787""","""Delayed""","""Chicago""","""2025-06-20 15:00:00""","""Seattle""","""2025-06-20 19:00:00"""
"""DL801""","""JetBlue""","""Boeing 787""","""Delayed""","""Seattle""","""2025-06-20 06:00:00""","""Miami""","""2025-06-20 10:00:00"""
…,…,…,…,…,…,…,…
"""AA426""","""American Airlines""","""Boeing 737""","""Cancelled""","""New York""","""2025-06-22 06:00:00""","""Los Angeles""","""2025-06-22 12:00:00"""
"""DL544""","""Delta""","""Embraer 190""","""On Time""","""Seattle""","""2025-06-20 06:00:00""","""London""","""2025-06-20 15:00:00"""
"""AA943""","""United""","""Embraer 190""","""On Time""","""New York""","""2025-06-19 18:00:00""","""Miami""","""2025-06-19 22:00:00"""


- Pasemos a `schema_overrides` un diccionario de columnas para convertir columnas de texto a columnas de fecha/hora.
- También ordenemos por `departure_time` de paso.

In [ ]:
pl.read_json(
    "flights.json",
    schema_overrides={"departure_time": pl.Datetime, "arrival_time": pl.Datetime},
).sort("departure_time")

flight_id,airline,aircraft,status,departure_city,departure_time,arrival_city,arrival_time
str,str,str,str,str,datetime[μs],str,datetime[μs]
"""DL961""","""Delta""","""Boeing 787""","""On Time""","""Chicago""",2025-06-15 08:00:00,"""Paris""",2025-06-15 12:00:00
"""DL677""","""Lufthansa""","""Airbus A320""","""Cancelled""","""New York""",2025-06-15 13:00:00,"""Los Angeles""",2025-06-15 19:00:00
"""UA204""","""United""","""Embraer 190""","""On Time""","""New York""",2025-06-15 14:00:00,"""Los Angeles""",2025-06-15 20:00:00
"""DL898""","""Lufthansa""","""Boeing 787""","""On Time""","""Miami""",2025-06-15 15:00:00,"""London""",2025-06-15 19:00:00
"""DL734""","""American Airlines""","""Boeing 787""","""Delayed""","""Tokyo""",2025-06-15 17:00:00,"""Miami""",2025-06-15 21:00:00
…,…,…,…,…,…,…,…
"""AA169""","""JetBlue""","""Boeing 737""","""Delayed""","""Miami""",2025-06-24 04:00:00,"""Tokyo""",2025-06-24 08:00:00
"""DL638""","""American Airlines""","""Embraer 190""","""Cancelled""","""Seattle""",2025-06-24 04:00:00,"""New York""",2025-06-24 08:00:00
"""AA791""","""JetBlue""","""Boeing 737""","""Cancelled""","""Tokyo""",2025-06-25 00:00:00,"""Los Angeles""",2025-06-25 11:00:00


El método `replace_time_zone` en Polars se utiliza para asignar una zona horaria a una columna de fecha y hora que actualmente es 'ingenua' (naive), es decir, que no tiene una zona horaria definida.

Aquí tienes los puntos clave de cómo funciona:

- **Asignación, no conversió**n: No cambia las horas ni los minutos de los datos existentes. Simplemente 'etiqueta' la columna con la zona horaria proporcionada (por ejemplo, "America/New_York").
- **Requisito previo**: La columna debe ser de tipo pl.Datetime. Si es una cadena de texto, primero debes convertirla usando str.to_datetime o schema_overrides.
- **Cambio de Tipo**: El tipo de dato de la columna cambiará de datetime[μs] a datetime[μs, Zona/Horaria].
- **Metadatos**: Una vez que la columna tiene una zona horaria, Polars puede manejar correctamente cambios de horario de verano (DST) y permite usar el método convert_time_zone si luego necesitas ver esos mismos instantes en otra parte del mundo

In [ ]:
pl.read_json(
    "flights.json",
    schema_overrides={"departure_time": pl.Datetime, "arrival_time": pl.Datetime},
).sort("departure_time").with_columns(
    pl.col("departure_time")
      .dt.replace_time_zone("America/New_York"),
    pl.col("arrival_time")
      .dt.replace_time_zone("America/New_York"),
)

flight_id,airline,aircraft,status,departure_city,departure_time,arrival_city,arrival_time
str,str,str,str,str,"datetime[μs, America/New_York]",str,"datetime[μs, America/New_York]"
"""DL961""","""Delta""","""Boeing 787""","""On Time""","""Chicago""",2025-06-15 08:00:00 EDT,"""Paris""",2025-06-15 12:00:00 EDT
"""DL677""","""Lufthansa""","""Airbus A320""","""Cancelled""","""New York""",2025-06-15 13:00:00 EDT,"""Los Angeles""",2025-06-15 19:00:00 EDT
"""UA204""","""United""","""Embraer 190""","""On Time""","""New York""",2025-06-15 14:00:00 EDT,"""Los Angeles""",2025-06-15 20:00:00 EDT
"""DL898""","""Lufthansa""","""Boeing 787""","""On Time""","""Miami""",2025-06-15 15:00:00 EDT,"""London""",2025-06-15 19:00:00 EDT
"""DL734""","""American Airlines""","""Boeing 787""","""Delayed""","""Tokyo""",2025-06-15 17:00:00 EDT,"""Miami""",2025-06-15 21:00:00 EDT
…,…,…,…,…,…,…,…
"""AA169""","""JetBlue""","""Boeing 737""","""Delayed""","""Miami""",2025-06-24 04:00:00 EDT,"""Tokyo""",2025-06-24 08:00:00 EDT
"""DL638""","""American Airlines""","""Embraer 190""","""Cancelled""","""Seattle""",2025-06-24 04:00:00 EDT,"""New York""",2025-06-24 08:00:00 EDT
"""AA791""","""JetBlue""","""Boeing 737""","""Cancelled""","""Tokyo""",2025-06-25 00:00:00 EDT,"""Los Angeles""",2025-06-25 11:00:00 EDT


### Lectura Adicional
- https://docs.pola.rs/user-guide/transformations/time-series/parsing/#mixed-offsets
- https://docs.pola.rs/user-guide/transformations/time-series/timezones/
- https://docs.pola.rs/api/python/stable/reference/api/polars.read_json.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.replace_time_zone.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.offset_by.html

## Zonas Horarias II: Conversiones
- La biblioteca estándar de Python incluye un módulo `zoneinfo`. Fue introducido en Python 3.9.
- La función `available_timezones` del módulo `zoneinfo` devuelve un conjunto de todas las zonas horarias.

In [ ]:
import zoneinfo

print(len(zoneinfo.available_timezones()))

timezones = zoneinfo.available_timezones()

599


- El conjunto `timezones` incluye tanto "UTC" como zonas horarias específicas de ciudades.

In [ ]:
"Europe/Paris" in timezones

True

- Podemos pasar al método `dt.replace_time_zone` cualquiera de estas zonas horarias.
- Supongamos que las fechas/horas de nuestro dataset están almacenadas basándose en la hora de Nueva York.
- El tipo de dato de la columna cambia. Observa que el valor formateado de cada fila también incluye `EDT`.

In [ ]:
flights = (
    pl.read_json(
        "flights.json",
        schema_overrides={"departure_time": pl.Datetime, "arrival_time": pl.Datetime},
    )
    .sort("departure_time")
    .with_columns(
        pl.col("departure_time").dt.replace_time_zone("Europe/Paris"),
        pl.col("arrival_time").dt.replace_time_zone("Europe/Paris"),
    )
)
flights

flight_id,airline,aircraft,status,departure_city,departure_time,arrival_city,arrival_time
str,str,str,str,str,"datetime[μs, Europe/Paris]",str,"datetime[μs, Europe/Paris]"
"""DL961""","""Delta""","""Boeing 787""","""On Time""","""Chicago""",2025-06-15 08:00:00 CEST,"""Paris""",2025-06-15 12:00:00 CEST
"""DL677""","""Lufthansa""","""Airbus A320""","""Cancelled""","""New York""",2025-06-15 13:00:00 CEST,"""Los Angeles""",2025-06-15 19:00:00 CEST
"""UA204""","""United""","""Embraer 190""","""On Time""","""New York""",2025-06-15 14:00:00 CEST,"""Los Angeles""",2025-06-15 20:00:00 CEST
"""DL898""","""Lufthansa""","""Boeing 787""","""On Time""","""Miami""",2025-06-15 15:00:00 CEST,"""London""",2025-06-15 19:00:00 CEST
"""DL734""","""American Airlines""","""Boeing 787""","""Delayed""","""Tokyo""",2025-06-15 17:00:00 CEST,"""Miami""",2025-06-15 21:00:00 CEST
…,…,…,…,…,…,…,…
"""AA169""","""JetBlue""","""Boeing 737""","""Delayed""","""Miami""",2025-06-24 04:00:00 CEST,"""Tokyo""",2025-06-24 08:00:00 CEST
"""DL638""","""American Airlines""","""Embraer 190""","""Cancelled""","""Seattle""",2025-06-24 04:00:00 CEST,"""New York""",2025-06-24 08:00:00 CEST
"""AA791""","""JetBlue""","""Boeing 737""","""Cancelled""","""Tokyo""",2025-06-25 00:00:00 CEST,"""Los Angeles""",2025-06-25 11:00:00 CEST


- El método `dt.convert_time_zone` convierte de una zona horaria a otra.
- Por ejemplo, Los Ángeles está 3 horas adelante de Nueva York.

In [ ]:
flights.select(
    pl.col("departure_time"),
    pl.col("departure_time")
    .dt.convert_time_zone("America/Los_Angeles")
    .alias("departure_time_on_west_coast"),
    pl.col("departure_time")
    .dt.convert_time_zone("Europe/London")
    .alias("departure_time_in_london"),
)

departure_time,departure_time_on_west_coast,departure_time_in_london
"datetime[μs, Europe/Paris]","datetime[μs, America/Los_Angeles]","datetime[μs, Europe/London]"
2025-06-15 08:00:00 CEST,2025-06-14 23:00:00 PDT,2025-06-15 07:00:00 BST
2025-06-15 13:00:00 CEST,2025-06-15 04:00:00 PDT,2025-06-15 12:00:00 BST
2025-06-15 14:00:00 CEST,2025-06-15 05:00:00 PDT,2025-06-15 13:00:00 BST
2025-06-15 15:00:00 CEST,2025-06-15 06:00:00 PDT,2025-06-15 14:00:00 BST
2025-06-15 17:00:00 CEST,2025-06-15 08:00:00 PDT,2025-06-15 16:00:00 BST
…,…,…
2025-06-24 04:00:00 CEST,2025-06-23 19:00:00 PDT,2025-06-24 03:00:00 BST
2025-06-24 04:00:00 CEST,2025-06-23 19:00:00 PDT,2025-06-24 03:00:00 BST
2025-06-25 00:00:00 CEST,2025-06-24 15:00:00 PDT,2025-06-24 23:00:00 BST


### Lectura Adicional
- https://docs.python.org/3/library/zoneinfo.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.replace_time_zone.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.convert_time_zone.html

---
# Ejercicios

### Ejercicio 1
Carga el archivo `clock_in_times.csv` parseando las fechas automáticamente. Luego, crea una nueva columna llamada `"duracion"` que represente la diferencia entre `clock_out` y `clock_in`. Finalmente, muestra la duración total en **minutos** con una columna llamada `"minutos_totales"`

### Ejercicio 2
Carga el archivo `history.csv` con parseo automático de fechas. Filtra las filas cuya fecha sea un **día laboral** (`is_business_day`) y que pertenezcan al **siglo 19** (años 1801–1900). Ordena el resultado por fecha ascendente.

### Ejercicio 3
Carga el archivo `deliveries.csv` y convierte las columnas `order_date` y `delivery_date` a tipo `Date` usando el formato `"%D"`. Luego, agrega las siguientes columnas:
- `"dias_entrega"`: número total de días entre el pedido y la entrega.
- `"mes_pedido"`: mes del pedido.
- `"dia_semana_entrega"`: día de la semana de la entrega (1=Lunes, 7=Domingo).

### Ejercicio 4
Carga el archivo `flights.json` con `schema_overrides` para que `departure_time` y `arrival_time` sean de tipo `Datetime`. Asigna la zona horaria `"Europe/Paris"` a ambas columnas y luego crea dos columnas nuevas:
- `"hora_salida_tokio"`: la hora de salida convertida a la zona horaria `"Asia/Tokyo"`.
- `"hora_llegada_ny"`: la hora de llegada convertida a la zona horaria `"America/New_York"`.

### Ejercicio 5
Carga `deliveries.csv` y convierte `order_date` a tipo `Datetime` con formato `"%D"`. Luego, usa `dt.offset_by` para crear las siguientes columnas:
- `"entrega_estimada"`: la fecha del pedido más 2 semanas (`"2w"`).
- `"siguiente_trimestre"`: la fecha del pedido más 3 meses calendario (`"3mo"`).

Finalmente, convierte ambas columnas nuevas a cadena de texto con el formato `"%d/%m/%Y %H:%M"` usando `dt.to_string`.